# Project Assignment 1

## Setup

In [1]:
master = "spark://spark-master:7077"   # use "spark://spark-master:7077" when running inside the Docker cluster
dataFolder = "/data/project/" if master.startswith("spark://") else "../data/"


In [2]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as sf
from pyspark.sql.types import StructType, StructField, IntegerType, StringType
from pyspark.sql import Window
import pyspark.sql.functions as sf

spark_builder = (SparkSession.builder
         .master(master)
         .appName("spark-project")
         .config("spark.driver.memory", "2g")
         .config("spark.executor.memory", "1536m")   # stay under the worker's --memory 2G cap
         .config("spark.ui.showConsoleProgress", "false"))

if master.startswith("spark://"):
    spark_builder = (spark_builder
                      .config("spark.driver.host", "jupyter")
                      .config("spark.driver.bindAddress", "0.0.0.0"))

spark = spark_builder.getOrCreate()

spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.conf.set("spark.sql.adaptive.skewJoin.enabled", "true")

print("Spark UI:", spark.sparkContext.uiWebUrl)
print("Default parallelism (cores seen by the driver):", spark.sparkContext.defaultParallelism)


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/06 13:29:06 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark UI: http://jupyter:4040
Default parallelism (cores seen by the driver): 6


In [3]:
import sys
sys.path.append("/workspace/project")

from spark_profiler import SparkProfiler

profiler = SparkProfiler(spark)


## M1

In [9]:
with profiler.profile("M1-baseline"):
    zones_df = (spark.read.option("header", "true")
                .option("inferSchema", "true")
                .csv(dataFolder + "taxi_zone_lookup.csv"))

    raw_df = spark.read.parquet(dataFolder + "yellow_tripdata_*.parquet")

    # Preparar aliases para as zonas para facilitar os joins
    pickup_zones = zones_df.select(
        sf.col("LocationID").alias("pu_loc_id"),
        sf.col("Borough").alias("pickup_borough"),
        sf.col("Zone").alias("pickup_zone")
    )

    # Base de dados limpa e enriquecida com colunas temporais e zonas já associadas
    base_df = (raw_df.filter(
            (sf.col("fare_amount") > 0) &
            (sf.col("trip_distance") > 0) &
            (sf.col("passenger_count") > 0) &
            (sf.col("passenger_count") <= 6) &
            (sf.col("tpep_dropoff_datetime") > sf.col("tpep_pickup_datetime")) &
            (sf.col("PULocationID").between(1, 263)) &
            (sf.col("DOLocationID").between(1, 263))
        )
        .withColumn("month", sf.month("tpep_pickup_datetime"))
        .withColumn("day_of_week", sf.date_format("tpep_pickup_datetime", "E"))
        .withColumn("hour_of_day", sf.hour("tpep_pickup_datetime"))
        .withColumn("trip_duration_mins", (sf.unix_timestamp("tpep_dropoff_datetime") - sf.unix_timestamp("tpep_pickup_datetime")) / 60.0)
        .join(pickup_zones, sf.col("PULocationID") == sf.col("pu_loc_id"), "inner")
    )

    base_df.count()

    print("partitions after repartition:", zones_df.rdd.getNumPartitions())
    print("partitions after repartition:", raw_df.rdd.getNumPartitions())



26/11/05 23:03:51 WARN FileStreamSink: Assume no metadata directory. Error while looking for metadata directory in the path: /data/project/yellow_tripdata_*.parquet.
java.io.FileNotFoundException: File /data/project/yellow_tripdata_*.parquet does not exist
	at org.apache.hadoop.fs.RawLocalFileSystem.deprecatedGetFileStatus(RawLocalFileSystem.java:980)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileLinkStatusInternal(RawLocalFileSystem.java:1301)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileStatus(RawLocalFileSystem.java:970)
	at org.apache.hadoop.fs.FilterFileSystem.getFileStatus(FilterFileSystem.java:462)
	at org.apache.spark.sql.execution.streaming.sinks.FileStreamSink$.hasMetadata(FileStreamSink.scala:58)
	at org.apache.spark.sql.execution.datasources.DataSource.resolveRelation(DataSource.scala:394)
	at org.apache.spark.sql.catalyst.analysis.ResolveDataSource.org$apache$spark$sql$catalyst$analysis$ResolveDataSource$$loadV1BatchSource(ResolveDataSource.scala:210)
	at org.ap

partitions after repartition: 1
partitions after repartition: 10
[M1-baseline] wall time: 4.83s (7 stages, 25 tasks) - see profiler.report() / profiler.compare(...)


In [73]:
base_df.printSchema()
ver quais as colunas que são precissas de base_df
passenger_count
trip_distance
RatecodeID
PULocationID
DOLocationID
payment_type
fare_amount
month
day_of_week
hour_of_day
trip_duration_mins
pickup_borough
pickup_zone
dropoff_borough

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)
 |-- cbd_congestion_fee: double (nullable = true)
 |-- month: integer (nullable = true)
 |-- day_of_week: string (nullable = t

In [10]:
with profiler.profile("M1-Schema"):
    zones_schema = StructType([
        StructField("LocationID", IntegerType(), False),
        StructField("Borough", StringType(), False),
        StructField("Zone", StringType(), False),
        StructField("service_zone", StringType(), False),
    ])

    zones_df = (spark.read.option("header", "true")
                .schema(zones_schema)
                .csv(dataFolder + "taxi_zone_lookup.csv"))

    raw_df = spark.read.parquet(dataFolder + "yellow_tripdata_*.parquet")

    # Preparar aliases para as zonas para facilitar os joins
    pickup_zones = zones_df.select(
        sf.col("LocationID").alias("pu_loc_id"),
        sf.col("Borough").alias("pickup_borough"),
        sf.col("Zone").alias("pickup_zone")
    )

    # Base de dados limpa e enriquecida com colunas temporais e zonas já associadas
    base_df = (raw_df.filter(
            (sf.col("fare_amount") > 0) &
            (sf.col("trip_distance") > 0) &
            (sf.col("passenger_count") > 0) &
            (sf.col("passenger_count") <= 6) &
            (sf.col("tpep_dropoff_datetime") > sf.col("tpep_pickup_datetime")) &
            (sf.col("PULocationID").between(1, 263)) &
            (sf.col("DOLocationID").between(1, 263))
        )
        .withColumn("month", sf.month("tpep_pickup_datetime"))
        .withColumn("day_of_week", sf.date_format("tpep_pickup_datetime", "E"))
        .withColumn("hour_of_day", sf.hour("tpep_pickup_datetime"))
        .withColumn("trip_duration_mins", (sf.unix_timestamp("tpep_dropoff_datetime") - sf.unix_timestamp("tpep_pickup_datetime")) / 60.0)
        .join(pickup_zones, sf.col("PULocationID") == sf.col("pu_loc_id"), "inner")
    )

    base_df.count()

26/11/05 23:03:58 WARN FileStreamSink: Assume no metadata directory. Error while looking for metadata directory in the path: /data/project/yellow_tripdata_*.parquet.
java.io.FileNotFoundException: File /data/project/yellow_tripdata_*.parquet does not exist
	at org.apache.hadoop.fs.RawLocalFileSystem.deprecatedGetFileStatus(RawLocalFileSystem.java:980)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileLinkStatusInternal(RawLocalFileSystem.java:1301)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileStatus(RawLocalFileSystem.java:970)
	at org.apache.hadoop.fs.FilterFileSystem.getFileStatus(FilterFileSystem.java:462)
	at org.apache.spark.sql.execution.streaming.sinks.FileStreamSink$.hasMetadata(FileStreamSink.scala:58)
	at org.apache.spark.sql.execution.datasources.DataSource.resolveRelation(DataSource.scala:394)
	at org.apache.spark.sql.catalyst.analysis.ResolveDataSource.org$apache$spark$sql$catalyst$analysis$ResolveDataSource$$loadV1BatchSource(ResolveDataSource.scala:210)
	at org.ap

[M1-Schema] wall time: 2.99s (5 stages, 23 tasks) - see profiler.report() / profiler.compare(...)


In [11]:
profiler.compare("M1-baseline", "M1-Schema")

,M1-baseline,M1-Schema,change
metric,,,
wall_time_s,4.83,2.99,-38.1%
num_jobs,6,4,-33.3%
num_stages,7,5,-28.6%
num_tasks,25,23,-8.0%
executor_run_time_ms,12874,10039,-22.0%
executor_cpu_time_ms,6566,4891,-25.5%
input_bytes,13.3MB,13.3MB,-0.2%
output_bytes,0.0B,0.0B,-
shuffle_read_bytes,590.0B,590.0B,+0.0%


In [16]:
with profiler.profile("M1-cache"):
    zones_schema = StructType([
        StructField("LocationID", IntegerType(), False),
        StructField("Borough", StringType(), False),
        StructField("Zone", StringType(), False),
        StructField("service_zone", StringType(), False),
    ])

    zones_df = (spark.read.option("header", "true")
                .schema(zones_schema)
                .csv(dataFolder + "taxi_zone_lookup.csv"))

    raw_df = spark.read.parquet(dataFolder + "yellow_tripdata_*.parquet")

    # Preparar aliases para as zonas para facilitar os joins
    pickup_zones = zones_df.select(
        sf.col("LocationID").alias("pu_loc_id"),
        sf.col("Borough").alias("pickup_borough"),
        sf.col("Zone").alias("pickup_zone")
    )

    # Base de dados limpa e enriquecida com colunas temporais e zonas já associadas
    base_df = (raw_df.filter(
            (sf.col("fare_amount") > 0) &
            (sf.col("trip_distance") > 0) &
            (sf.col("passenger_count") > 0) &
            (sf.col("passenger_count") <= 6) &
            (sf.col("tpep_dropoff_datetime") > sf.col("tpep_pickup_datetime")) &
            (sf.col("PULocationID").between(1, 263)) &
            (sf.col("DOLocationID").between(1, 263))
        )
        .withColumn("month", sf.month("tpep_pickup_datetime"))
        .withColumn("day_of_week", sf.date_format("tpep_pickup_datetime", "E"))
        .withColumn("hour_of_day", sf.hour("tpep_pickup_datetime"))
        .withColumn("trip_duration_mins", (sf.unix_timestamp("tpep_dropoff_datetime") - sf.unix_timestamp("tpep_pickup_datetime")) / 60.0)
        .join(pickup_zones, sf.col("PULocationID") == sf.col("pu_loc_id"), "inner")
        .cache()
    )

    base_df.count()

26/11/05 23:06:49 WARN FileStreamSink: Assume no metadata directory. Error while looking for metadata directory in the path: /data/project/yellow_tripdata_*.parquet.
java.io.FileNotFoundException: File /data/project/yellow_tripdata_*.parquet does not exist
	at org.apache.hadoop.fs.RawLocalFileSystem.deprecatedGetFileStatus(RawLocalFileSystem.java:980)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileLinkStatusInternal(RawLocalFileSystem.java:1301)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileStatus(RawLocalFileSystem.java:970)
	at org.apache.hadoop.fs.FilterFileSystem.getFileStatus(FilterFileSystem.java:462)
	at org.apache.spark.sql.execution.streaming.sinks.FileStreamSink$.hasMetadata(FileStreamSink.scala:58)
	at org.apache.spark.sql.execution.datasources.DataSource.resolveRelation(DataSource.scala:394)
	at org.apache.spark.sql.catalyst.analysis.ResolveDataSource.org$apache$spark$sql$catalyst$analysis$ResolveDataSource$$loadV1BatchSource(ResolveDataSource.scala:210)
	at org.ap

[M1-cache] wall time: 53.78s (6 stages, 33 tasks) - see profiler.report() / profiler.compare(...)


In [4]:
with profiler.profile("M1-Select"):
    zones_schema = StructType([
        StructField("LocationID", IntegerType(), False),
        StructField("Borough", StringType(), False),
        StructField("Zone", StringType(), False),
        StructField("service_zone", StringType(), False),
    ])

    zones_df = (spark.read.option("header", "true")
                .schema(zones_schema)
                .csv(dataFolder + "taxi_zone_lookup.csv"))

    raw_df = spark.read.parquet(dataFolder + "yellow_tripdata_*.parquet")

    # Preparar aliases para as zonas para facilitar os joins
    pickup_zones = zones_df.select(
        sf.col("LocationID").alias("pu_loc_id"),
        sf.col("Borough").alias("pickup_borough"),
        sf.col("Zone").alias("pickup_zone")
    )

    # Base de dados limpa e enriquecida com colunas temporais e zonas já associadas
    base_df = (raw_df.filter(
            (sf.col("fare_amount") > 0) &
            (sf.col("trip_distance") > 0) &
            (sf.col("passenger_count") > 0) &
            (sf.col("passenger_count") <= 6) &
            (sf.col("tpep_dropoff_datetime") > sf.col("tpep_pickup_datetime")) &
            (sf.col("PULocationID").between(1, 263)) &
            (sf.col("DOLocationID").between(1, 263))
        )
        .withColumn("month", sf.month("tpep_pickup_datetime"))
        .withColumn("day_of_week", sf.date_format("tpep_pickup_datetime", "E"))
        .withColumn("hour_of_day", sf.hour("tpep_pickup_datetime"))
        .withColumn("trip_duration_mins", (sf.unix_timestamp("tpep_dropoff_datetime") - sf.unix_timestamp("tpep_pickup_datetime")) / 60.0)
        .join(pickup_zones.hint("broadcast"), sf.col("PULocationID") == sf.col("pu_loc_id"), "inner")
        .select("passenger_count", "trip_distance", "RatecodeID","PULocationID", "DOLocationID", "payment_type", "fare_amount", "month", "day_of_week", "hour_of_day", "trip_duration_mins", "pickup_borough", "pickup_zone")
        .cache()
    )

    base_df.count()

26/11/06 13:29:22 WARN FileStreamSink: Assume no metadata directory. Error while looking for metadata directory in the path: /data/project/yellow_tripdata_*.parquet.
java.io.FileNotFoundException: File /data/project/yellow_tripdata_*.parquet does not exist
	at org.apache.hadoop.fs.RawLocalFileSystem.deprecatedGetFileStatus(RawLocalFileSystem.java:980)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileLinkStatusInternal(RawLocalFileSystem.java:1301)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileStatus(RawLocalFileSystem.java:970)
	at org.apache.hadoop.fs.FilterFileSystem.getFileStatus(FilterFileSystem.java:462)
	at org.apache.spark.sql.execution.streaming.sinks.FileStreamSink$.hasMetadata(FileStreamSink.scala:58)
	at org.apache.spark.sql.execution.datasources.DataSource.resolveRelation(DataSource.scala:394)
	at org.apache.spark.sql.catalyst.analysis.ResolveDataSource.org$apache$spark$sql$catalyst$analysis$ResolveDataSource$$loadV1BatchSource(ResolveDataSource.scala:210)
	at org.ap

[M1-Select] wall time: 34.20s (6 stages, 33 tasks) - see profiler.report() / profiler.compare(...)


In [14]:
base_df.unpersist()

DataFrame[passenger_count: bigint, trip_distance: double, RatecodeID: bigint, PULocationID: int, DOLocationID: int, payment_type: bigint, fare_amount: double, month: int, day_of_week: string, hour_of_day: int, trip_duration_mins: double, pickup_borough: string, pickup_zone: string]

In [ ]:
passenger_count
trip_distance
RatecodeID
PULocationID
DOLocationID
payment_type
fare_amount
month
day_of_week
hour_of_day
trip_duration_mins
pickup_borough
pickup_zone

## M2

In [14]:
spark.conf.set("spark.sql.adaptive.enabled", "false")
spark.conf.set("spark.sql.adaptive.skewJoin.enabled", "false")
with profiler.profile("M2-without_skew"):
    o2 = (base_df
            .groupBy("pickup_borough", "day_of_week", "hour_of_day")
            .agg(
                sf.count("*").alias("total_trip_count"),
                sf.avg("fare_amount").alias("mean_fare_amount"),
                sf.avg("trip_duration_mins").alias("mean_trip_duration"),
                sf.avg("trip_distance").alias("mean_trip_distance")
            )
        )

    o2.write.mode("overwrite").format("noop").save()

[M2-without_skew] wall time: 2.90s (2 stages, 210 tasks) - see profiler.report() / profiler.compare(...)


In [12]:
o2.explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- HashAggregate(keys=[pickup_borough#843, day_of_week#848, hour_of_day#849], functions=[count(1), avg(fare_amount#832), avg(trip_duration_mins#850), avg(trip_distance#826)])
   +- Exchange hashpartitioning(pickup_borough#843, day_of_week#848, hour_of_day#849, 200), ENSURE_REQUIREMENTS, [plan_id=471]
      +- HashAggregate(keys=[pickup_borough#843, day_of_week#848, hour_of_day#849], functions=[partial_count(1), partial_avg(fare_amount#832), partial_avg(trip_duration_mins#850), partial_avg(trip_distance#826)])
         +- InMemoryTableScan [trip_distance#826, fare_amount#832, day_of_week#848, hour_of_day#849, trip_duration_mins#850, pickup_borough#843]
               +- InMemoryRelation [passenger_count#825L, trip_distance#826, RatecodeID#827L, PULocationID#829, DOLocationID#830, payment_type#831L, fare_amount#832, month#847, day_of_week#848, hour_of_day#849, trip_duration_mins#850, pickup_borough#843, pickup_zone#844], StorageLeve

In [15]:
profiler.compare("M2-baseline", "M2-without_skew")

,M2-baseline,M2-without_skew,change
metric,,,
wall_time_s,2.27,2.9,+27.8%
num_jobs,2,1,-50.0%
num_stages,3,2,-33.3%
num_tasks,21,210,+900.0%
executor_run_time_ms,8705,10839,+24.5%
executor_cpu_time_ms,8564,10503,+22.6%
input_bytes,1.1GB,1.1GB,+0.0%
output_bytes,0.0B,0.0B,-
shuffle_read_bytes,595.3KB,595.3KB,+0.0%


## M3

In [43]:
with profiler.profile("M3-M1_Cache_With_Select"):
    zone_month_stats = (base_df
        .groupBy("pickup_zone", "pickup_borough", "month")
            .agg(
                sf.avg("fare_amount").alias("mean_fare"),
                sf.stddev("fare_amount").alias("stddev_fare"),
                sf.avg("trip_distance").alias("mean_distance"),
                sf.stddev("trip_distance").alias("stddev_distance"),
                sf.count("*").alias("monthly_trip_volume")
        )
    )

    w_borough_month = Window.partitionBy("pickup_borough", "month")
    w_zone_rolling = Window.partitionBy("pickup_borough", "pickup_zone").orderBy("month").rowsBetween(-2, 0)
    w_borough_rank = Window.partitionBy("pickup_borough", "month")
    w_borough_rolling_time = Window.partitionBy("pickup_borough").orderBy("month").rowsBetween(-2, 0)

    f = (zone_month_stats
        .withColumn("borough_monthly_volume", sf.sum("monthly_trip_volume").over(w_borough_month))
        .withColumn("volume_rank", sf.rank().over(w_borough_rank.orderBy(sf.desc("monthly_trip_volume"))))
        .withColumn("rolling_3m_zone_volume", sf.sum("monthly_trip_volume").over(w_zone_rolling))
        .withColumn("rolling_3m_borough_volume", sf.sum("borough_monthly_volume").over(w_borough_rolling_time))
        .withColumn("rolling_3m_trip_share", sf.col("rolling_3m_zone_volume") / sf.col("rolling_3m_borough_volume"))
        .withColumn("rolling_mean_volume", sf.avg("monthly_trip_volume").over(w_zone_rolling))
        .withColumn("rolling_stddev_volume", sf.stddev("monthly_trip_volume").over(w_zone_rolling))
        .withColumn("is_volume_deviating", sf.when(
            sf.col("rolling_stddev_volume").isNotNull() & (sf.col("rolling_stddev_volume") > 0),
            sf.abs(sf.col("monthly_trip_volume") - sf.col("rolling_mean_volume")) > (1.5 * sf.col("rolling_stddev_volume"))
            ).otherwise(False)
        )
    )

    f.show()

+--------------------+--------------+-----+------------------+------------------+------------------+------------------+-------------------+----------------------+-----------+----------------------+-------------------------+---------------------+-------------------+---------------------+-------------------+
|         pickup_zone|pickup_borough|month|         mean_fare|       stddev_fare|     mean_distance|   stddev_distance|monthly_trip_volume|borough_monthly_volume|volume_rank|rolling_3m_zone_volume|rolling_3m_borough_volume|rolling_3m_trip_share|rolling_mean_volume|rolling_stddev_volume|is_volume_deviating|
+--------------------+--------------+-----+------------------+------------------+------------------+------------------+-------------------+----------------------+-----------+----------------------+-------------------------+---------------------+-------------------+---------------------+-------------------+
|Allerton/Pelham G...|         Bronx|    1|42.001234567901236|14.47714141178

In [44]:
profiler.compare("M3-M1_Cache", "M3-M1_Cache_With_Select")


,M3-M1_Cache,M3-M1_Cache_With_Select,change
metric,,,
wall_time_s,6.88,6.78,-1.5%
num_jobs,4,4,+0.0%
num_stages,10,10,+0.0%
num_tasks,46,46,+0.0%
executor_run_time_ms,18554,14471,-22.0%
executor_cpu_time_ms,16878,13949,-17.4%
input_bytes,2.5GB,1.1GB,-55.2%
output_bytes,0.0B,0.0B,-
shuffle_read_bytes,757.2KB,757.3KB,+0.0%


## O1

In [45]:
with profiler.profile("O1-M1_Cache_With_Select"):
    o1_base = base_df.withColumn(
        "fare_per_dist", sf.col("fare_amount") / sf.col("trip_distance")
    ).withColumn(
        "is_weekend",
        sf.when(sf.col("day_of_week").isin("Sat", "Sun", "Sáb", "Dom"), sf.lit("weekend"))
        .otherwise(sf.lit("weekday"))
    )

    stats_df = o1_base.groupBy("PULocationID", "hour_of_day").agg(
        sf.mean("fare_per_dist").alias("mean_fare_per_dist"),
        sf.stddev_samp("fare_per_dist").alias("std_fare_per_dist")
    )

    anomalies_df = o1_base.join(
        stats_df, ["PULocationID", "hour_of_day"], "inner"
    ).withColumn(
        "is_anomaly",
        sf.when(
            (sf.col("std_fare_per_dist").isNotNull()) &
            (sf.col("std_fare_per_dist") > 0) &
            (sf.col("fare_per_dist") > (sf.col("mean_fare_per_dist") + 3.0 * sf.col("std_fare_per_dist"))),
            1
        ).otherwise(0)
    )

    o1_result = anomalies_df.groupBy(
        "pickup_borough",
        "payment_type",
        "is_weekend"
    ).agg(
        sf.count("*").alias("total_trips"),
        sf.sum("is_anomaly").alias("anomaly_count"),
        (sf.sum("is_anomaly") / sf.count("*")).alias("anomaly_rate")
    )

    o1_result.show()

+--------------+------------+----------+-----------+-------------+--------------------+
|pickup_borough|payment_type|is_weekend|total_trips|anomaly_count|        anomaly_rate|
+--------------+------------+----------+-----------+-------------+--------------------+
|        Queens|           2|   weekday|     376287|         1512|0.004018209504978912|
|     Manhattan|           3|   weekend|      35661|          455|0.012759036482431789|
|           EWR|           2|   weekday|         43|            1|0.023255813953488372|
| Staten Island|           3|   weekday|          8|            0|                 0.0|
|         Bronx|           3|   weekday|        257|           39|  0.1517509727626459|
| Staten Island|           1|   weekend|         46|            2|0.043478260869565216|
|           EWR|           1|   weekday|        672|            8|0.011904761904761904|
|     Manhattan|           1|   weekend|    7039379|         3676|5.222051547444739E-4|
|      Brooklyn|           3|   

In [46]:
profiler.compare("O1-M1_Cache", "O1-M1_Cache_With_Select")

,O1-M1_Cache,O1-M1_Cache_With_Select,change
metric,,,
wall_time_s,17.06,15.57,-8.7%
num_jobs,5,5,+0.0%
num_stages,9,9,+0.0%
num_tasks,67,67,+0.0%
executor_run_time_ms,60698,53769,-11.4%
executor_cpu_time_ms,45967,48625,+5.8%
input_bytes,3.8GB,2.2GB,-41.4%
output_bytes,0.0B,0.0B,-
shuffle_read_bytes,401.1MB,401.1MB,+0.0%


## O4

In [49]:
with profiler.profile("O4-M1_Cache_With_Select"):
    enriched_df = (base_df
            .withColumn("is_airport", sf.when(sf.col("RatecodeID").isin([2, 3]), 1).otherwise(0))
            .withColumn("is_negotiated", sf.when(sf.col("RatecodeID") == 5, 1).otherwise(0))
            .withColumn("is_group", sf.when(sf.col("passenger_count") >= 4, 1).otherwise(0)))

    bucket_stats_df = enriched_df.groupBy("PULocationID", "hour_of_day").agg(
        sf.count("*").alias("total_bucket_trips"),
        sf.avg("is_airport").alias("frac_airport"),
        sf.avg("is_negotiated").alias("frac_negotiated"),
        sf.avg("is_group").alias("frac_group")
    )

    classified_buckets_df = bucket_stats_df.withColumn(
        "zone_type",
        sf.when(sf.col("frac_airport") > 0.3, "transit")
          .when(sf.col("frac_negotiated") > 0.2, "commercial")
          .when(sf.col("frac_group") > 0.15, "residential")
          .otherwise("mixed")
    )

    final_o4_df = enriched_df.join(
        classified_buckets_df.select("PULocationID", "hour_of_day", "zone_type"),
        on=["PULocationID", "hour_of_day"],
        how="inner"
    )

    o4_monthly_stats = final_o4_df.groupBy("zone_type", "month").agg(
        sf.avg("fare_amount").alias("mean_fare"),
        sf.avg("trip_distance").alias("mean_distance"),
        sf.count("*").alias("total_trips")
    )

    o4_monthly_stats.show()

+-----------+-----+------------------+------------------+-----------+
|  zone_type|month|         mean_fare|     mean_distance|total_trips|
+-----------+-----+------------------+------------------+-----------+
|residential|   12| 46.70188679245283| 8.992830188679244|         53|
|      mixed|    5|17.279903676530186| 3.350282578072457|    2987953|
|    transit|   12| 61.35747665781927|15.087328040099695|     142446|
|      mixed|   12|18.086607208313342|2.8457119566197098|    2766250|
|    transit|    6| 61.82751012893602|15.230066816539379|     159691|
|      mixed|    6|17.138869891346534|2.7367204792594526|    2714854|
| commercial|   12| 53.89777393681155| 6.127396896856676|      12439|
|    transit|    5| 62.37492174311421|15.465449766073293|     165864|
| commercial|    6|51.106473617724426| 5.682101836556221|      10291|
|      mixed|   11| 17.61812796924969|2.8066460797478694|    2821209|
|residential|    6| 32.48181818181819| 4.499090909090909|         11|
|    transit|   11| 

In [50]:
profiler.compare("O4-M1_Cache", "O4-M1_Cache_With_Select")

,O4-M1_Cache,O4-M1_Cache_With_Select,change
metric,,,
wall_time_s,14.1,7.76,-45.0%
num_jobs,5,5,+0.0%
num_stages,9,9,+0.0%
num_tasks,68,68,+0.0%
executor_run_time_ms,46680,27997,-40.0%
executor_cpu_time_ms,34601,25241,-27.1%
input_bytes,4.5GB,2.2GB,-50.7%
output_bytes,0.0B,0.0B,-
shuffle_read_bytes,443.9MB,443.9MB,+0.0%


In [ ]:
spark.stop()